In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import warnings # Ignore specific warnings
warnings.filterwarnings("ignore")

In [2]:
df1 = pd.read_excel('output_lube_oil_g11.xlsx')

In [3]:
# انتخاب ستون‌ها برای استانداردسازی
data_to_scale = df1[['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287']]

# استانداردسازی داده‌ها
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)

# تبدیل خروجی به دیتافریم با همان نام ستون‌ها
scaled_df = pd.DataFrame(scaled_data, columns=['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287'])


In [4]:
scaled_df_clean = scaled_df.dropna()

In [5]:
# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# اضافه کردن لیبل‌ها به دیتافریم
df = scaled_df_clean.copy()
df['label'] = labels

# جدا کردن داده‌های نویز و خوشه‌ها
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1

noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله هر نویز از نزدیک‌ترین نقطه در خوشه‌ها
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)

# نرمال‌سازی فاصله‌ها به بازه 0 تا 1
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت سری وزن ناهنجاری برای همه داده‌ها
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights

# اضافه کردن به دیتافریم نهایی
df['anomaly_weight'] = anomaly_weights


In [6]:
# train_model.py
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import joblib
import warnings

warnings.filterwarnings("ignore")

# بارگذاری داده‌ها
df1 = pd.read_excel('output_lube_oil_g11.xlsx')
selected_columns = ['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
                    'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
data_to_scale = df1[selected_columns]

# استانداردسازی
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)
scaled_df = pd.DataFrame(scaled_data, columns=selected_columns)
scaled_df_clean = scaled_df.dropna()

# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# جدا کردن داده‌های نویز و خوشه‌ها
df = scaled_df_clean.copy()
df['label'] = labels
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1
noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله و وزن ناهنجاری
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت بردار وزن ناهنجاری
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights
df['anomaly_weight'] = anomaly_weights

# ذخیره مدل‌ها و داده‌های مرجع
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(dbscan, 'dbscan_model.pkl')
np.save('cluster_points.npy', cluster_points)


In [8]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  # اگر رکوردی نبود، مقدار None قرار می‌گیره

# قرار دادن مقادیر در ۷ متغیر جداگانه
value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



✅ مقادیر آخرین رکوردها برای UnitID=11:
AssetID 8341 → Value: 0.07
AssetID 8342 → Value: 12.1
AssetID 8343 → Value: 69.0
AssetID 8344 → Value: -240.0
AssetID 8346 → Value: 6.9
AssetID 9286 → Value: 7.2
AssetID 9287 → Value: 1.28
{'is_anomaly': False, 'anomaly_weight': 0.0}


In [ ]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'Trusted_Connection=yes;'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  


value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



import mysql.connector
from datetime import datetime

# اتصال به دیتابیس MySQL
conn = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password='',  
    database='dsas'
)

cursor = conn.cursor()

# مقادیر ورودی
inputs = [value_8341,value_8342,value_8343,value_8344,value_8346,value_9286,value_9287]
anomaly_weight = result['anomaly_weight']  # مقدار score
results = "Normal" if anomaly_weight < 5 else "Abnormal"
model_name = "Anomaly detection for lube oil system"
unitID = 11
system = "dbscan clustering weighted by computing distance from clusters "
score = anomaly_weight
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
updated_at = created_at


query = """
    INSERT INTO results_dsas_mhi_lube_oil_11 
    (inputs, results, model_name, unitID, system, score, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""


cursor.execute(query, (
    str(inputs),  # تبدیل لیست به رشته برای ذخیره در فیلد text یا varchar
    results,
    model_name,
    unitID,
    system,
    score,
    created_at,
    updated_at
))

# ذخیره تغییرات
conn.commit()

print("✅ داده با موفقیت ثبت شد.")

# بستن اتصال
cursor.close()
conn.close()


✅ مقادیر آخرین رکوردها برای UnitID=11:
AssetID 8341 → Value: 0.07
AssetID 8342 → Value: 12.1
AssetID 8343 → Value: 69.0
AssetID 8344 → Value: -240.0
AssetID 8346 → Value: 6.9
AssetID 9286 → Value: 7.2
AssetID 9287 → Value: 1.28
{'is_anomaly': False, 'anomaly_weight': 0.0}
✅ داده با موفقیت ثبت شد.


In [ ]:
# pip install mysql-connector-python